# 09 · Deep unknown-fault test (Q3): LOCO-retrained CNN-6 and the three-way decision rule

## Purpose
Q3 asks whether the system notices a fault type it was never trained on. Each of the 9 fault classes is withheld in turn
(leave-one-class-out, LOCO); a CNN-6 (vibration only) is retrained without it, and the question is whether its recordings
are flagged **unknown** rather than passed off as a known fault or as healthy. The three-way rule turns scores into a
verdict: **healthy**, **a known fault**, or **unknown**. All numbers are **development findings** on the development pool,
not reserved-test results; one training seed.

## Protocol
```
for each withheld class c (9) and outer dev fold k (3):                       27 runs
  known classes (≠ c), folds ≠ k ──(saved Stage 2D.1 Q3 split, 4:1:1 within each folder)──┐
      ├── fit          → train CNN-6 (9 outputs) and, on the fit NORMALS only, AE-6 (frame level)
      │                   norm_stats in development mode on fit (context excludes everything below and all of c)
      ├── calibration  → thresholds τ_p (MSP), τ_d (Mahalanobis): 97.5 % marginal targets; τ_e = max calibration-normal AE error
      └── seen-folder evaluation (a)
  fold k: known classes (b, unseen folders; normal is within-folder) and the withheld class c   → scored once per c
```
- **Scores** (directions fixed before running): MSP (lower = more unknown); Mahalanobis distance in the 128-d embedding
  (class centroids from fit windows, one shared Ledoit–Wolf covariance of within-class residuals, minimum over known
  classes per window, mean over 9 windows); k-NN (cosine distance to the 10th nearest fit window, mean over windows);
  AE reconstruction error. *AE error measures departure from healthy behaviour, not unfamiliarity among fault types; a
  familiar severe fault can have a larger error than an unfamiliar mild one.*
- **AE per (c, k), not per k:** the saved Q3 assignment splits the normal folder differently for every withheld class, so
  each run retrains its own AE-6 on its own fit normals (CLAUDE.md §7).
- **Three-way rule, PRIMARY = classifier-first** (declared before running; the AE misses some mild faults, and AE-first
  would declare them healthy): confident (MSP ≥ τ_p and d ≤ τ_d) → healthy (top class normal) or that known fault;
  otherwise unknown, except top class normal with AE error ≤ τ_e → healthy. **AE-first** is a comparison only.
- **Classical comparison:** logistic regression / MSP of Stage 2D.1 on the same split and recordings, re-run per
  recording and checked against its saved per-fold results. Observed differences only; no winner claims without paired
  intervals.

Stage 3A/3B (Q2-only) embeddings are not used here.

## Inputs
`results/loco/*.json|csv`, `results/scores/cnn6_w-<c>_s42_loco_fold<k>_recording.parquet`,
`results/scores/cnn6ae6_s42_loco_decisions.parquet`, `results/scores/logreg_s42_loco_decisions.parquet`, MLflow
runs (`stage = 3C`), all written by `scripts/09_loco_cnn.py`.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code> and the Stage 3C results, and stops if they were produced under different split assignments.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.cache import check_split_hash
from ml.config import load_config
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, RunPaths
from ml.splits import FAULT_CLASSES

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
RES = SYSTEM_CODE_DIR / "results" / "loco"
FIG = SYSTEM_CODE_DIR / "results" / "figures"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
auc = json.loads((RES / "loco_auroc_oscr.json").read_text())
rule = json.loads((RES / "decision_rule.json").read_text())
for name, d in (("loco_auroc_oscr.json", auc), ("decision_rule.json", rule)):
    check_split_hash(d["provenance"]["split_hash"], meta, name)
dec = pd.read_parquet(RunPaths("loco", "cnn6ae6", 42, protocol="loco").scores("decisions"))
lr = pd.read_parquet(RunPaths("loco", "logreg", 42, protocol="loco").scores("decisions"))
per_fold = pd.read_csv(RES / "loco_per_fold_auroc_oscr.csv")
print(auc["training"])
print("LR reproduction exact:", rule["lr_reproduction"]["exact"])

## 1. Score distributions: fold-k known vs the withheld class
For each withheld class (rows) and score (columns): the fold-k known recordings (grey) against the withheld recordings
(colour), pooled over the three folds. Pooling mixes three fold models, so this figure is descriptive; the AUROC table
below is computed within fold.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws, for every withheld class and every score, histograms of the known fold-k recordings (grey) and of the withheld class (colour). MSP is shown as 1 − MSP on a log axis so that, like the other scores, larger means more unknown.</div>

In [ ]:
scores = {"1 − MSP": lambda d: np.clip(1 - d.msp, 1e-7, None), "Mahalanobis": lambda d: d.maha,
          "k-NN (cosine, 10th)": lambda d: np.clip(d.knn, 1e-6, None), "AE error": lambda d: d.ae}
fig, axes = plt.subplots(len(FAULT_CLASSES), 4, figsize=(15, 2.0 * len(FAULT_CLASSES)))
for i, c in enumerate(FAULT_CLASSES):
    g = dec[dec.withheld == c]
    for j, (name, f) in enumerate(scores.items()):
        ax = axes[i, j]
        kn, un = f(g[g.role == "foldk_known"]), f(g[g.role == "foldk_withheld"])
        lo, hi = min(kn.min(), un.min()), max(kn.max(), un.max())
        bins = np.logspace(np.log10(lo), np.log10(hi), 30)
        ax.hist(kn, bins=bins, color="grey", alpha=0.6, density=True)
        ax.hist(un, bins=bins, color="C3", alpha=0.6, density=True)
        ax.set_xscale("log")
        ax.xaxis.set_minor_formatter(plt.NullFormatter())
        ax.tick_params(labelsize=6)
        if i == 0:
            ax.set_title(name, fontsize=9)
        if j == 0:
            ax.set_ylabel(c.replace("_misalignment", "_mis"), fontsize=8)
fig.suptitle("fold-k known (grey) vs withheld class (red), pooled over folds; density", y=1.0)
fig.tight_layout()
fig.savefig(FIG / "09_loco_score_distributions.png")

## 2. Threshold-free results: AUROC and OSCR
Per withheld class: (fold-k known) vs (fold-k withheld), mean ± std over the three folds; macro = mean over the 9
classes. OSCR uses the CNN's closed-set correctness on the known recordings.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds the AUROC and OSCR tables: one row per withheld class, one column per score, fold mean ± std; then the macro summary per score.</div>

In [ ]:
pc = pd.read_csv(RES / "loco_per_class_auroc_oscr.csv")
fmt = lambda m, s: f"{m:.3f} ± {s:.3f}"
display(pc.assign(v=[fmt(m, s) for m, s in zip(pc.auroc_mean, pc.auroc_std)]).pivot(index="withheld", columns="score", values="v")
        [["msp", "maha", "knn", "ae"]].rename_axis(None, axis=1).rename(columns=lambda x: f"AUROC {x}"))
display(pc.assign(v=[fmt(m, s) for m, s in zip(pc.oscr_mean, pc.oscr_std)]).pivot(index="withheld", columns="score", values="v")
        [["msp", "maha", "knn", "ae"]].rename_axis(None, axis=1).rename(columns=lambda x: f"OSCR {x}"))
pd.DataFrame(auc["summary"]).T.round(3)

## 3. Measured joint acceptance
τ_p and τ_d are each set for 97.5 % marginal acceptance on the calibration known recordings; the joint acceptance (both
accepted) is not assumed but measured on calibration, seen-folder evaluation and unseen-folder (fold-k) known recordings.
Mean and std over the 27 runs.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates the measured acceptance of MSP alone, Mahalanobis alone and both together, for calibration, seen-folder and unseen-folder known recordings, split into all known, known faults and normal.</div>

In [ ]:
ja = pd.DataFrame(rule["joint_acceptance"]["per_group_mean_over_runs"])
ja["group"] = pd.Categorical(ja.group, ["calibration", "seen_folder_eval", "unseen_foldk"])
ja.sort_values(["group", "subset"]).set_index(["group", "subset"]).round(3)

## 4. Outcomes, PRIMARY ordering (classifier-first)
Rates at the calibrated thresholds. Withheld recordings: healthy / known fault / unknown (faults called healthy are
listed explicitly). Known faults: seen-folder (a) and unseen-folder fold k (b), healthy / correct / wrong fault / unknown,
macro over known fault classes. Normal recordings: healthy / fault / unknown.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Defines helpers that turn the saved outcome summaries into tables, then shows the primary (classifier-first) outcomes: per withheld class, overall, fold spread, known faults (a) and (b), and normal recordings.</div>

In [ ]:
def withheld_table(o):
    t = pd.DataFrame(o["withheld"]["per_class"]).T
    t.loc["overall (recordings)"] = pd.Series(o["withheld"]["overall_recordings"])
    t.loc["macro over classes"] = pd.Series(o["withheld"]["macro_over_classes"])
    t.loc["fold spread (std over runs)"] = pd.Series(o["withheld"]["fold_spread_std"])
    return t[["healthy", "known_fault", "unknown", "n"]]


def known_table(o):
    rows = {}
    for grp in ("a_seen_folder", "b_unseen_foldk"):
        d = o["known_faults"][grp]
        rows[(grp, "macro")] = d["macro_over_known_fault_classes"]
        rows[(grp, "fold spread (std)")] = d["fold_spread_std_of_run_macro"]
    return pd.DataFrame(rows).T[["healthy", "correct", "wrong_fault", "unknown"]]


def show(o, title):
    print(title)
    display(withheld_table(o).round(3))
    print("withheld faults called healthy:", o["withheld"]["faults_called_healthy_count"])
    display(known_table(o).round(3))
    display(pd.DataFrame(o["normal"]).T.round(3))


o1 = rule["outcomes"]["classifier_first"]
show(o1, "PRIMARY: classifier-first")
pd.DataFrame({g: {c: v["unknown"] for c, v in o1["known_faults"][g]["per_class"].items()}
              for g in ("a_seen_folder", "b_unseen_foldk")}).round(3).rename(columns=lambda x: f"false unknown {x}")

### AE override (classifier-first)
How often the AE exception changes *unknown* to *healthy* (top class normal, not confident, AE error ≤ τ_e), per fold,
separately for truly healthy, known-fault and withheld-fault recordings. An override on a fault recording is a fault
declared healthy by the tie-break.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates the AE overrides per fold and recording kind (counts and percentages), for fold-k and seen-folder recordings.</div>

In [ ]:
ov = pd.DataFrame(rule["override_counts"]["per_fold"])
ov.pivot_table(index=["role", "kind"], columns="outer_fold", values=["overrides", "n"], aggfunc="sum")

## 5. Comparison ordering (AE-first)
Reported for comparison only; the primary ordering was declared before running.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the same outcome tables for the AE-first ordering.</div>

In [ ]:
show(rule["outcomes"]["ae_first"], "COMPARISON: AE-first")

## 6. Classical comparison (logistic regression / MSP, Stage 2D.1)
Same split, same recordings. Within-fold AUROC and OSCR differences for MSP (CNN − LR), and the practical outcomes at
each method's own thresholds (LR: MSP at its 95 % calibration target; CNN: the primary three-way rule; supplementary:
CNN MSP alone at 95 %). Observed differences; no winner claims without paired intervals.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Plots the per-run AUROC difference (CNN MSP − LR MSP) for every withheld class and fold, then tabulates the practical outcomes side by side: share of withheld faults passing as a known fault, declared healthy or unknown, and the false-unknown rates (a), (b) with the folder gap.</div>

In [ ]:
d = pd.read_csv(RES / "cnn_minus_lr_msp_per_fold.csv")
fig, ax = plt.subplots(figsize=(10, 3.6))
for k in range(3):
    g = d[d.outer_fold == k].set_index("withheld").loc[list(FAULT_CLASSES)]
    ax.scatter(np.arange(9) + (k - 1) * 0.15, g.auroc_diff, label=f"fold {k}", s=25)
ax.axhline(0, color="k", lw=0.7)
ax.set_xticks(range(9), [c.replace("_misalignment", "_mis") for c in FAULT_CLASSES], rotation=30, ha="right", fontsize=8)
ax.set(ylabel="AUROC difference (CNN − LR), MSP", title="Within-fold MSP AUROC difference per withheld class and fold")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "09_loco_cnn_minus_lr.png")
cc = rule["classical_comparison"]
print({k: cc["auroc_msp_cnn_minus_lr"][k] for k in ("mean", "std", "runs_positive", "runs")})
pd.DataFrame({"CNN three-way (primary)": cc["practical_primary"], "CNN three-way (AE-first)": cc["practical_ae_first"],
              "LR / MSP (Stage 2D.1)": cc["practical_classical_lr"],
              "CNN MSP only, 95 % (suppl.)": cc["practical_cnn_msp_only_95"]}).round(3)

## Findings
Development findings (27 LOCO runs: 9 withheld classes × 3 folds, one seed); not reserved-test results.
- **Embedding distances separate unknown faults best; MSP less so; AE error does not.** Macro AUROC over the 9 withheld
  classes (fold-k known vs withheld): Mahalanobis 0.877 ± 0.194, k-NN 0.845 ± 0.174, MSP 0.782 ± 0.192, AE error
  0.517 ± 0.324 (single runs 0.40–1.00, 0.44–1.00, 0.18–0.99, 0.06–1.00). Imbalance (Mahalanobis 0.64) and vertical
  misalignment (0.45) are not separated; the six bearing classes are (0.95–0.995). The AE result is expected: its error
  measures departure from healthy behaviour, not unfamiliarity among fault types.
- **Joint acceptance is measured, not assumed.** With 97.5 % marginal calibration targets for τ_p and τ_d, the measured
  joint acceptance of known recordings is 0.988 on calibration, 0.972 on seen-folder evaluation and 0.864 on unseen-folder
  fold-k recordings: the folder shift again costs about 11 points.
- **Primary rule (classifier-first), withheld faults:** 60 % unknown (macro over classes 70 %), **36 % passed as a known
  fault**, 4 % declared healthy (28 recordings, 22 of them horizontal misalignment). Imbalance (97 % passed as known) and
  vertical misalignment (87 %) are the failures; underhang outer race is always flagged unknown.
- **Known faults:** false unknowns 2.3 % seen-folder (a) vs 13.6 % unseen-folder (b), a folder gap of 11 points; no known
  fault was declared healthy; wrong-fault calls ≤ 0.1 %. Normal: 85 % healthy on fold k, 15 % unknown, none called a fault.
- **AE override:** it turned unknown into healthy for 10 of 299 truly healthy recordings, 1 of 8,319 known-fault
  recordings and 9 of 779 withheld-fault recordings (faults declared healthy by the tie-break). τ_e rests on only 2–3
  calibration normals per run.
- **AE-first (comparison only):** fewer withheld faults called healthy (2.3 %), but 2.9–3.8 % of known faults are
  declared healthy (none under the primary rule) and 23–25 % of normal recordings become unknown (primary: 8–15 %). The
  primary ordering was fixed before running and is not changed.
- **Classical LR/MSP (same split and recordings), observed differences:** MSP AUROC CNN − LR −0.061 ± 0.182 over the 27
  runs (CNN higher in 12). At each method's own thresholds the CNN rule has fewer false unknowns ((a) 2.3 % vs 4.2 %,
  (b) 13.6 % vs 23.7 %; folder gap 11 vs 20 points) but passes more withheld faults as known (36 % vs 26 %); healthy
  4 % vs 4 %. No winner is claimed without paired intervals.


## Outputs written
- `results/figures/09_loco_score_distributions.png`, `09_loco_cnn_minus_lr.png`
- (by `scripts/09_loco_cnn.py`) `results/loco/*`; `results/scores/cnn6_w-<c>_s42_loco_fold<k>_*`,
  `cnn6ae6_s42_loco_decisions.parquet`, `logreg_s42_loco_decisions.parquet`; `artifacts/loco/<name>/fold<k>.pt`;
  MLflow runs tagged `stage = 3C`.